# This notebook goes through the preprocessing of the database so that is SBERT friendly.

## 0. Imports

In [22]:
import pandas as pd
import re
import unicodedata

## 1. Helpers

In [23]:
WS_RE = re.compile(r"\s+")
PUNCT_BAD = re.compile(r"[^\w\s\-\_]")
EDGE_DASH_UND = re.compile(r"(^[\-_]+|[\-_]+$)")

def nfkc(s: str) -> str:
    return unicodedata.normalize("NFKC", s)

def basic_clean(s: str) -> str:
    if pd.isna(s):
        return ""
    s = nfkc(str(s)).lower().strip()
    s = PUNCT_BAD.sub(" ", s)
    s = s.replace("/", " ")
    s = s.replace("&", " and ")
    s = s.replace("+", " plus ")
    s = s.replace("·", " ")
    s = s.replace("-", " ")
    s = s.replace("_", " ")
    s = WS_RE.sub(" ", s).strip()
    s = EDGE_DASH_UND.sub("", s).strip()
    return s

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Clean source & applications
    df["source_clean"] = df["source"].apply(basic_clean)
    df["application_clean"] = df["application"].apply(basic_clean)

    # Split functions into separate columns
    df["functions_split"] = df["functions"].fillna("").apply(
        lambda x: [basic_clean(f) for f in re.split(r"[;，、,]+", str(x)) if f.strip()]
    )

    # Expand into multiple columns (Function_1, Function_2, …)
    max_funcs = df["functions_split"].map(len).max()
    for i in range(max_funcs):
        df[f"function_{i+1}"] = df["functions_split"].apply(
            lambda lst: lst[i] if i < len(lst) else ""
        )

    # Build SBERT-friendly text by concatenating cleaned pieces
    df["sbert_text"] = (
        df["application_clean"]
        + df.apply(
            lambda row: " " + " ".join([f for f in row["functions_split"] if f]), axis=1
        )
        + (" " + df["source_clean"]).where(df["source_clean"] != "", "")
    ).str.strip()

    # Deduplicate on source_clean and application_clean
    df = df.drop_duplicates(subset=["source_clean"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["application_clean"]).reset_index(drop=True)

    # Reorder + drop helper
    func_cols = [f"function_{i+1}" for i in range(max_funcs)]
    #cols = ["source", "functions", "application",
    #        "source_clean", "application_clean"] + func_cols + ["sbert_text"]
    cols = ["source_clean"] + func_cols + ["application_clean"]
    return df[cols]

## 2. Load data

In [24]:
df_original = pd.read_csv('../data/materials_data_final.csv')
df_original

,source,functions,application
0,013 Denim,recycle-yarn; weave-fabric; support-innovation...,large work of art presented to the Dutch royal...
1,100% bacterial dye,produce pigments; create sustainable alternati...,microbial colour library for dyeing textiles
2,Basalt knitted fabric,reinforce-fabric; prevent-algal growth; extend...,reinforcement fabric for maritime applications
3,100% Biobased Flax panel,provide structural support; reduce environment...,interior wall panels
4,100% rejects waxed printed cotton,recycle-textiles; create-carpets; reduce-waste...,high-quality recycled carpets
...,...,...,...
3158,Zero furniture panel,reduce-CO2 emissions; sequester-carbon; provid...,furniture components for interior design
3159,Zero,reduce-joint thickness; provide-ventilation; a...,joint-free brick wall construction
3160,Zinc Foam,absorb-energy; increase-strength; create-foam;...,energy-absorbing structural component
3161,Zintek – Titanium zinc,enhance-properties; develop-patina; provide-du...,architectural facades


## 3. Preprocess

In [25]:
df_clean = preprocess(df_original)
df_clean.to_csv('materials_data_preprocessed.csv',index=False)
df_clean

,source_clean,function_1,function_2,function_3,function_4,function_5,function_6,application_clean
0,013 denim,recycle yarn,weave fabric,support innovation,connect community,,,large work of art presented to the dutch royal...
1,100 bacterial dye,produce pigments,create sustainable alternative,reduce water usage,minimize energy consumption,,,microbial colour library for dyeing textiles
2,basalt knitted fabric,reinforce fabric,prevent algal growth,extend residence time,conduct heat poorly,resist electricity,,reinforcement fabric for maritime applications
3,100 biobased flax panel,provide structural support,reduce environmental impact,enable biobased composition,,,,interior wall panels
4,100 rejects waxed printed cotton,recycle textiles,create carpets,reduce waste,innovate design,,,high quality recycled carpets
...,...,...,...,...,...,...,...,...
2805,zero furniture panel,reduce co2 emissions,sequester carbon,provide bending strength,facilitate coating,enable sawing,,furniture components for interior design
2806,zero,reduce joint thickness,provide ventilation,absorb rainwater,slow ageing,prevent water penetration,,joint free brick wall construction
2807,zinc foam,absorb energy,increase strength,create foam,cast lightweight structure,,,energy absorbing structural component
2808,zintek titanium zinc,enhance properties,develop patina,provide durability,offer longevity,reduce maintenance,,architectural facades
